In [1]:
import xarray as xr
xr.set_options(display_style="text")
import zarr
import dask.array as dsa

In [6]:
globalHackathonStoreNamesList

['ngc4008_P1D_0.zarr',
 'ngc4008_P1D_1.zarr',
 'ngc4008_P1D_2.zarr',
 'ngc4008_P1D_3.zarr',
 'ngc4008_P1D_4.zarr',
 'ngc4008_P1D_5.zarr',
 'ngc4008_P1D_6.zarr',
 'ngc4008_P1D_7.zarr',
 'ngc4008_P1D_8.zarr',
 'ngc4008_P1D_9.zarr',
 'ngc4008_P1D_10.zarr',
 'ngc4008_PT3H_0.zarr',
 'ngc4008_PT3H_1.zarr',
 'ngc4008_PT3H_2.zarr',
 'ngc4008_PT3H_3.zarr',
 'ngc4008_PT3H_4.zarr',
 'ngc4008_PT3H_5.zarr',
 'ngc4008_PT3H_6.zarr',
 'ngc4008_PT3H_7.zarr',
 'ngc4008_PT3H_8.zarr',
 'ngc4008_PT3H_9.zarr',
 'ngc4008_PT3H_10.zarr',
 'ngc4008_P15M_0.zarr',
 'ngc4008_P15M_1.zarr',
 'ngc4008_P15M_2.zarr',
 'ngc4008_P15M_3.zarr',
 'ngc4008_P15M_4.zarr',
 'ngc4008_P15M_5.zarr',
 'ngc4008_P15M_6.zarr',
 'ngc4008_P15M_7.zarr',
 'ngc4008_P15M_8.zarr',
 'ngc4008_P15M_9.zarr',
 'ngc4008_P15M_10.zarr']

In [3]:
ds = xr.tutorial.open_dataset("air_temperature")
# create initial chunk structure
ds = ds.chunk({"time": 100})
ds.air.encoding = {}  # helps when writing to zarr
ds

<xarray.Dataset> Size: 31MB
Dimensions:  (time: 2920, lat: 25, lon: 53)
Coordinates:
  * time     (time) datetime64[ns] 23kB 2013-01-01 ... 2014-12-31T18:00:00
  * lat      (lat) float32 100B 75.0 72.5 70.0 67.5 65.0 ... 22.5 20.0 17.5 15.0
  * lon      (lon) float32 212B 200.0 202.5 205.0 207.5 ... 325.0 327.5 330.0
Data variables:
    air      (time, lat, lon) float64 31MB dask.array<chunksize=(100, 25, 53), meta=np.ndarray>
Attributes:
    Conventions:  COARDS
    title:        4x daily NMC reanalysis (1948)
    description:  Data is from NMC initialized reanalysis\n(4x/day).  These a...
    platform:     Model
    references:   http://www.esrl.noaa.gov/psd/data/gridded/data.ncep.reanaly...

In [4]:
ds.air.data

dask.array<xarray-air, shape=(2920, 25, 53), dtype=float64, chunksize=(100, 25, 53), chunktype=numpy.ndarray>

# Delete any zarr store in this directory and create a new zarr store

In [5]:
! rm -rf *.zarr # clean up any existing temporary data
ds.to_zarr("air_temperature.zarr")

In [6]:
source_group = zarr.open("air_temperature.zarr")
print(source_group.tree())

/
 ├── air (2920, 25, 53) float64
 ├── lat (25,) float32
 ├── lon (53,) float32
 └── time (2920,) float32


In [7]:
source_array = source_group["air"]
source_array.info

Name,/air
Type,zarr.core.Array
Data type,float64
Shape,"(2920, 25, 53)"
Chunk shape,"(100, 25, 53)"
Order,C
Read-only,False
Compressor,"Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,30952000 (29.5M)
No. bytes stored,17699107 (16.9M)


# Create a rechunker object

In [8]:
from rechunker import rechunk

target_chunks = (2920, 25, 1)
max_mem = "12MB"

target_store = "air_rechunked.zarr"
temp_store = "air_rechunked-tmp.zarr"

array_plan = rechunk(
    source_array, target_chunks, max_mem, target_store, temp_store=temp_store
)
array_plan

<Rechunked>
* Source      : <zarr.core.Array '/air' (2920, 25, 53) float64>

* Intermediate: <zarr.core.Array (2920, 25, 53) float64>

* Target      : <zarr.core.Array (2920, 25, 53) float64>

# Specifying target as a dictionary

In [10]:
target_chunks_dict = {"time": 2920, "lat": 25, "lon": 1}

# need to remove the existing stores or it won't work
!rm -rf air_rechunked.zarr air_rechunked-tmp.zarr
array_plan = rechunk(
    source_array, target_chunks_dict, max_mem, target_store, temp_store=temp_store
)
array_plan

<Rechunked>
* Source      : <zarr.core.Array '/air' (2920, 25, 53) float64>

* Intermediate: <zarr.core.Array (2920, 25, 53) float64>

* Target      : <zarr.core.Array (2920, 25, 53) float64>

In [11]:
result = array_plan.execute()
result.chunks

(2920, 25, 1)

In [12]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    array_plan.execute()

[########################################] | 100% Completed | 303.75 ms


In [13]:
target_array_dask = dsa.from_zarr("air_rechunked.zarr")
target_array_dask

dask.array<from-zarr, shape=(2920, 25, 53), dtype=float64, chunksize=(2920, 25, 1), chunktype=numpy.ndarray>

In [14]:
import os
from minio import Minio
from minio.error import S3Error

In [15]:
client = Minio("s3.eu-dkrz-3.dkrz.cloud",
    access_key=os.getenv("ACCESS_KEY_S3_nextGEMS"),
    secret_key=os.getenv("SECRET_KEY_S3_nextGEMS"),
)

In [16]:
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

nextgems 2024-11-20 08:01:03.841000+00:00


In [17]:
objects = client.list_objects("nextgems")
for obj in objects:
    print(obj.object_name)

myjfs/
ngc4008_P1D_0.zarr/
ngc4008_P1D_1.zarr/
ngc4008_P1D_10.zarr/
ngc4008_P1D_2.zarr/
ngc4008_P1D_3.zarr/
ngc4008_P1D_4.zarr/
ngc4008_P1D_5.zarr/
ngc4008_P1D_6.zarr/
ngc4008_P1D_7.zarr/
ngc4008_P1D_8.zarr/
ngc4008_P1D_9.zarr/
ngc4008_PT3H_6.zarr/
rechunked_ngc4008/
recursive-copy/
test/


In [18]:
objects = client.list_objects("nextgems")
for obj in objects:
    print(obj.object_name)

myjfs/
ngc4008_P1D_0.zarr/
ngc4008_P1D_1.zarr/
ngc4008_P1D_10.zarr/
ngc4008_P1D_2.zarr/
ngc4008_P1D_3.zarr/
ngc4008_P1D_4.zarr/
ngc4008_P1D_5.zarr/
ngc4008_P1D_6.zarr/
ngc4008_P1D_7.zarr/
ngc4008_P1D_8.zarr/
ngc4008_P1D_9.zarr/
ngc4008_PT3H_6.zarr/
rechunked_ngc4008/
recursive-copy/
test/
